# Compare four LLM APIs

How do different models answer the same question?
We will ask about a Swiss voting initiative, first in English and then in German.

You will write four short API functions. The prompts and comparison code are provided.
Later, you will add a source text and try the OpenAI Decisions API.

Work through one section at a time. Complete each `TODO` when you reach it.
You can check your work in `llm_intro_solution.ipynb`.

We use the historical initiative **“Für Ehe und Familie – gegen die Heiratsstrafe”**,
which was voted on **28 February 2016**. Keep this date when asking about it to avoid
confusing it with later proposals about marriage taxation.

## Get ready

Use Python 3.12. Open a terminal in this repository and create a virtual environment:

```sh
python -m venv .venv
```

Activate it using the command for your system.

**Windows PowerShell**

```powershell
.venv\Scripts\Activate.ps1
```

**macOS or Linux**

```sh
source .venv/bin/activate
```

Then install the packages:

```sh
python -m pip install -r requirements.txt
```

### Add your keys and open the notebook

Copy `.env.example` to a new file named `.env`.
Fill in the four API keys. Keep this file private.
The model names are already supplied.

Start JupyterLab:

```sh
python -m jupyterlab
```

Open `llm_intro.ipynb` and select the `.venv` Python kernel.
Run the cells in order.

### Load the libraries and settings

Run the next two cells. They load the API keys and model names from `.env`.
You do not need to change this code.

Check that each provider prints `key configured: True` before continuing.

In [ ]:
import os

import requests
import anthropic
from dotenv import load_dotenv
from google import genai
from openai import OpenAI

In [ ]:
load_dotenv(".env")

google_api_key = os.getenv("GOOGLE_API_KEY") or os.getenv("GEMINI_API_KEY")
openai_api_key = os.getenv("OPENAI_API_KEY")
deepinfra_api_key = os.getenv("DEEPINFRA_API_KEY")
anthropic_api_key = os.getenv("ANTHROPIC_API_KEY")

google_model = os.getenv("GOOGLE_MODEL", "gemini-3.8-flash")
openai_model = os.getenv("OPENAI_MODEL", "gpt-6-luna")
deepinfra_model = os.getenv("DEEPINFRA_MODEL", "deepseek-ai/DeepSeek-V4.1-Flash")
anthropic_model = os.getenv("ANTHROPIC_MODEL", "claude-haiku-5-5")

# Show whether keys are present, without printing the keys themselves.
for provider, key in [("Google", google_api_key), ("OpenAI", openai_api_key),
                      ("DeepInfra", deepinfra_api_key), ("Anthropic", anthropic_api_key)]:
    print(provider, "key configured:", bool(key))

## The question we will ask

Read the two prompts below. They ask the same question in English and German.
Run the cell to save them as `prompt_en` and `prompt_de`.

We will reuse these variables so every provider receives exactly the same text
within each language.

In [ ]:
prompt_en = """Explain the Swiss popular initiative "Für Ehe und Familie – gegen die
Heiratsstrafe", voted on 28 February 2016, to someone unfamiliar with it.
What did it propose about taxes, social insurance and the definition of marriage?
Answer neutrally in English, using at most 120 words. Say if you do not know a detail."""

prompt_de = """Erkläre die Schweizer Volksinitiative «Für Ehe und Familie – gegen die
Heiratsstrafe», über die am 28. Februar 2016 abgestimmt wurde, für jemanden ohne Vorwissen.
Was schlug sie zu Steuern, Sozialversicherungen und zur Definition der Ehe vor?
Antworte neutral auf Deutsch mit höchstens 120 Wörtern. Sage, wenn du ein Detail nicht weisst."""

## Connect to the four providers

Each function follows the same pattern:

1. Create a client using the API key.
2. Send the prompt to the selected model.
3. Return the answer as a string.

`return` lets us save the answer and compare it later. The comparison code will print it.

### 1. Google Gemini

Pass the prompt as `contents`. The answer is in `response.text`.

[API examples](https://ai.google.dev/api/generate-content)

Complete `ask_google(prompt)` below.

1. Create the client with `genai.Client(api_key=google_api_key)`.
2. Call `client.models.generate_content(...)` with `model=google_model`
   and `contents=prompt`. Save the result as `response`.
3. Return `response.text`.

In [ ]:
def ask_google(prompt):
    # TODO: send prompt to this provider and return the answer as a string.
    raise NotImplementedError("Implement ask_google")

### 2. OpenAI

Here we use the Responses API. Pass the prompt as `input`.
The answer is in `response.output_text`.

[API examples](https://developers.openai.com/api/docs/guides/text)

Complete `ask_openai(prompt)` below.

1. Create the client with `OpenAI(api_key=openai_api_key)`.
2. Call `client.responses.create(...)` with `model=openai_model`
   and `input=prompt`. Save the result as `response`.
3. Return `response.output_text`.

In [ ]:
def ask_openai(prompt):
    # TODO: send prompt to this provider and return the answer as a string.
    raise NotImplementedError("Implement ask_openai")

### 3. DeepInfra

DeepInfra works with the OpenAI Python package, using its own key and API address.
This call sends the prompt as a user message.

[API examples](https://docs.deepinfra.com/chat/overview)

Complete `ask_deepinfra(prompt)` below.

1. Create an `OpenAI` client using `api_key=deepinfra_api_key` and
   `base_url="https://api.deepinfra.com/v1/openai"`.
2. Call `client.chat.completions.create(...)` with `model=deepinfra_model`
   and the user message below. Save the result as `response`.

```python
messages=[{"role": "user", "content": prompt}]
```

3. Return `response.choices[0].message.content`. This is the first answer's text.

In [ ]:
def ask_deepinfra(prompt):
    # TODO: send prompt to this provider and return the answer as a string.
    raise NotImplementedError("Implement ask_deepinfra")

### 4. Anthropic

Anthropic also receives a user message. Its answer comes back in several blocks.
Keep the blocks whose type is `"text"` and join their text into one string.

[API examples](https://platform.claude.com/docs/en/api/messages/create)

Complete `ask_anthropic(prompt)` below.

1. Create the client with `anthropic.Anthropic(api_key=anthropic_api_key)`.
2. Call `client.messages.create(...)` with `model=anthropic_model`,
   `max_tokens=4096` and the user message below. Save the result as `response`.

```python
messages=[{"role": "user", "content": prompt}]
```

3. Collect `block.text` from the blocks in `response.content`
   where `block.type == "text"`.
4. Join those strings with `"\n"` and return the result.

`max_tokens=4096` sets the output budget. It leaves room for the model's
internal reasoning as well as its short answer.

In [ ]:
def ask_anthropic(prompt):
    # TODO: send prompt to this provider and return the answer as a string.
    raise NotImplementedError("Implement ask_anthropic")

## Compare the answers

The function below is provided. It sends `prompt` to each of the four API functions
and prints their answers.

It takes only one argument: the prompt text. Each API function already selects its model.
For example, `compare_models(prompt_en)` uses the English prompt.

Run this cell to define the function. It will make requests when we call it below.

In [ ]:
def compare_models(prompt):
    answers = {}

    print("\nGoogle Gemini")
    answers["Google Gemini"] = ask_google(prompt)
    print(answers["Google Gemini"])

    print("\nOpenAI")
    answers["OpenAI"] = ask_openai(prompt)
    print(answers["OpenAI"])

    print("\nDeepInfra")
    answers["DeepInfra"] = ask_deepinfra(prompt)
    print(answers["DeepInfra"])

    print("\nAnthropic")
    answers["Anthropic"] = ask_anthropic(prompt)
    print(answers["Anthropic"])

    return answers

### A. Ask without a source text

Start with English. Run the next cell and read the four answers before continuing.

Each comparison sends four API requests. Re-running a cell sends them again
and may incur charges. The full notebook makes 26 requests.

In [ ]:
answers_en = compare_models(prompt_en)

Now run the German version. Do the models describe the same proposals in both languages?

In [ ]:
answers_de = compare_models(prompt_de)

### B. Give the models a source text

Read the two snippets below. They summarize the proposal voted on in 2016.
They do not describe current law or give voting results.

These are teaching summaries based on the
[Federal Council's description](https://www.admin.ch/de/fur-ehe-und-familie).

In [ ]:
source_text_en = """The 2016 initiative sought to prevent disadvantages for married
couples in taxation and social insurance compared with other living arrangements.
It proposed treating spouses as one economic unit for tax purposes and defining
marriage as a lasting union between a man and a woman. The Federal Council and
Parliament recommended rejecting the initiative."""

source_text_de = """Die Initiative von 2016 wollte Benachteiligungen von Ehepaaren
bei Steuern und Sozialversicherungen gegenüber anderen Lebensformen verhindern.
Sie wollte Ehepaare steuerlich als eine Wirtschaftsgemeinschaft behandeln und die
Ehe als dauerhafte Verbindung von Mann und Frau definieren. Bundesrat und Parlament
empfahlen, die Initiative abzulehnen."""

We will keep the original question and add the matching snippet.
We also ask the models to use only the supplied text.

Run the next cell and read the completed English prompt that it prints.

In [ ]:
prompt_with_text_en = (
    f"{prompt_en}\n\n"
    "Use only the source text below. If a detail is missing, say it is not in the text.\n\n"
    f"Source text:\n{source_text_en}"
)

prompt_with_text_de = (
    f"{prompt_de}\n\n"
    "Verwende nur den folgenden Quelltext. Wenn ein Detail fehlt, sage, dass es nicht im Text steht.\n\n"
    f"Quelltext:\n{source_text_de}"
)

print(prompt_with_text_en)

Run the English comparison. Which details in the answers can you find in the snippet?

In [ ]:
answers_with_text_en = compare_models(prompt_with_text_en)

Repeat with the German prompt and snippet.

In [ ]:
answers_with_text_de = compare_models(prompt_with_text_de)

### How this prepares us for RAG

Here, we chose a passage and put it into the prompt ourselves.

Later, **retrieval-augmented generation (RAG)** will find relevant passages in documents
and add them to the prompt. The model then answers using that information.
We have just practised this last step.

### C. Ask about missing information

The snippets do not give the percentage of people who voted yes.
What should a model say if it must answer using only this text?

Read the two prompts below, then run the cell to save them.

In [ ]:
missing_prompt_en = (
    "What percentage voted yes on the initiative on 28 February 2016? "
    "Answer in English using only the source text. "
    "If the answer is missing, say it is not in the text.\n\n"
    f"Source text:\n{source_text_en}"
)

missing_prompt_de = (
    "Wie hoch war der Ja-Stimmenanteil zur Initiative am 28. Februar 2016? "
    "Antworte auf Deutsch und verwende nur den Quelltext. "
    "Wenn die Antwort fehlt, sage, dass sie nicht im Text steht.\n\n"
    f"Quelltext:\n{source_text_de}"
)

Try the English question. Check whether each answer acknowledges that the percentage is missing.

In [ ]:
missing_answers_en = compare_models(missing_prompt_en)

Now try it in German. Does any model supply a figure despite the instruction to use only the text?

In [ ]:
missing_answers_de = compare_models(missing_prompt_de)

## Pause and compare

Write a short note in each column. Mention differences between English and German
where you notice them.

| Provider | Without text | With text | Missing information |
| --- | --- | --- | --- |
| Google Gemini | | | |
| OpenAI | | | |
| DeepInfra | | | |
| Anthropic | | | |

Discuss these questions:

- What changed when you supplied the text?
- Did changing the language change any facts?
- Did the models admit when information was missing?
- Which answers would you check before sharing them with someone else?

You can read a saved answer again without making another request:

```python
answers_with_text_en["DeepInfra"]
```

If a call fails, check that provider's API key, credits and model access.
Then re-run the comparison cell.

## One more API: OpenAI Decisions

So far, we have asked models to write answers. Now we will ask two yes/no questions
about a snippet:

1. Does it say spouses would be treated as one economic unit for taxes?
2. Does it give the percentage who voted yes?

The [Decisions API](https://developers.openai.com/api/docs/guides/decisions)
can return a probability for each question. A value near 1 means the model thinks
the answer is yes; a value near 0 means it thinks the answer is no.

This kind of check is called a `predicate`. Read the two questions below and run the cell.
The same questions will be used for both snippets.

In [ ]:
decision_questions = [
    {
        "type": "predicate",
        "name": "tax_unit",
        "instructions": (
            "Does the text say the initiative proposed treating spouses "
            "as one economic unit for tax purposes?"
        ),
    },
    {
        "type": "predicate",
        "name": "vote_percentage",
        "instructions": (
            "Does the text give the percentage who voted yes? "
            "Use only the supplied text."
        ),
    },
]

### Send the snippet and the questions

We use `requests.post` to send a JSON request directly to the API.
This also works with OpenAI SDK versions that do not yet include Decisions.
The request uses the same OpenAI key as before.

For this endpoint, use `gpt-6-luna`. The function returns a list of answer dictionaries,
so its result has a different format from the four text-generation functions.

Complete `ask_openai_decisions(source_text)` below.

Send `requests.post(...)` with these arguments:

- URL: `"https://api.openai.com/v1/decisions"`
- `headers`: `{"Authorization": f"Bearer {openai_api_key}"}`
- `timeout`: `60` seconds
- `json`: the dictionary shown below

```python
{
    "model": "gpt-6-luna",
    "input": source_text,
    "questions": decision_questions,
}
```

Save the result as `response`. Then:

1. Call `response.raise_for_status()` to report an unsuccessful request.
2. Return `response.json()["answers"]` to get the list of answers.

In [ ]:
def ask_openai_decisions(source_text):
    # TODO: send the snippet and both predicate questions to the Decisions API.
    # Return the list of answers from the JSON response.
    raise NotImplementedError("Implement ask_openai_decisions")

### Read the results

Before running the English snippet, predict which question should get a high probability.
Then run the cell. Each printed dictionary is one answer:

- `name` identifies the question.
- `type` should be `predicate`.
- `probability` is the model's estimate.

If the type is `refusal`, the API declined the check and did not give a probability.

In [ ]:
decision_answers_en = ask_openai_decisions(source_text_en)
for answer in decision_answers_en:
    print(answer)

Repeat with the German snippet. Do the results agree with your reading of both texts?
A probability is still a model estimate, so check it against the snippet.

In [ ]:
decision_answers_de = ask_openai_decisions(source_text_de)
for answer in decision_answers_de:
    print(answer)